In [1]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder,LabelEncoder,StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC 
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import VotingClassifier,StackingClassifier,RandomForestClassifier
import seaborn as sns
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
import pickle as pkl

# Data Collection

In [2]:
df = pd.read_csv("train.csv")
df=df.drop(["Name","Ticket","Cabin","PassengerId"],axis=1)
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [3]:
df.isnull().sum()

Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

# Preprocessing

In [4]:
num_statergy = SimpleImputer(strategy="mean")
cat_statergy = SimpleImputer(strategy="most_frequent")

In [5]:
num_col = num_statergy.fit_transform(df[['Age']])
cat_col = cat_statergy.fit_transform(df[["Embarked"]])

In [6]:
df["Age"]= num_col.ravel()
df["Embarked"] = cat_col.ravel()

# Feature Engineering

In [7]:
ohe = OneHotEncoder(drop="first",sparse_output=False)
ohe_data=ohe.fit_transform(df[['Embarked']])

In [8]:
ohe_data = pd.DataFrame(ohe_data, columns=ohe.get_feature_names_out(["Embarked"]))

In [9]:
df=pd.concat([df,ohe_data],axis=1)

In [10]:
df=df.drop("Embarked",axis=1)

In [11]:
le = LabelEncoder()
le_data = le.fit_transform(df['Sex'])
df["Sex"] = le_data

# Train-Test Split

In [12]:
X = df.drop("Survived",axis=1)
y= df.Survived

In [13]:
X_train, X_test, y_train, y_test=train_test_split(X,y,test_size=0.3,random_state=42)

# Hyperparameter Tuning

In [14]:
models = {
    "LogisticRegression" : {
        "model":LogisticRegression(),
        "params":{
            "C" : [0.1,10,11,50,100],
            "max_iter":[100,1000,500]
        }
    },
    "SVC" : {
        "model": SVC(),
        "params":{
            "C" : [0.1,10,11,50,100],
            "kernel":['rbf',"linear"],
        }
    },
"KNeighborsClassifier" : {
    "model":KNeighborsClassifier(),
    "params": {
    "n_neighbors":[3,5,11,7,9]
    }
},
    "RandomForestClassifier" : {
        "model":RandomForestClassifier(),
        "params":{
             "n_estimators":[13,15,11,42],
            "criterion" : ["gini", "entropy", "log_loss"]
        
        }
    }
    
}

In [ ]:
best_params = []
for model_name,md in models.items():
    GScv = GridSearchCV(md["model"],md["params"])
    pipe = Pipeline([("scale",StandardScaler()),("GScv",GScv)])
    pipe.fit(X_train,y_train)
    best_params.append({
        "model":model_name,
        "score":pipe.named_steps["GScv"].best_score_,
        "params": pipe.named_steps["GScv"].best_params_,
    })

best_params = pd.DataFrame(best_params)    
best_params

## Model Training

# 1) LogisticRegression

In [15]:
log_reg = LogisticRegression(C= 0.1, max_iter= 100)
pipe = Pipeline([("scale",StandardScaler()),("log_reg",log_reg)])
pipe.fit(X_train,y_train)
y_pred = pipe.predict(X_test)
accuracy_score(y_test,y_pred)

0.8059701492537313

# 2) SVC

In [16]:
svc = SVC(C=50,)
pipe = Pipeline([("scale",StandardScaler()),("svc",svc)])
pipe.fit(X_train,y_train)
y_pred = pipe.predict(X_test)
accuracy_score(y_test,y_pred)

0.7910447761194029

# 3) knn

In [17]:
knn = KNeighborsClassifier(n_neighbors= 7)
pipe = Pipeline([("scale",StandardScaler()),("knn",knn)])
pipe.fit(X_train,y_train)
y_pred = pipe.predict(X_test)
accuracy_score(y_test,y_pred)

0.7798507462686567

# 4) RandomForestClassifier

In [18]:
rfc = RandomForestClassifier(criterion= 'entropy', n_estimators= 11)
pipe = Pipeline([("scale",StandardScaler()),("rfc",rfc)])
pipe.fit(X_train,y_train)
y_pred = pipe.predict(X_test)
accuracy_score(y_test,y_pred)

0.7798507462686567

# 5) VotingClassifier

In [19]:
vc = VotingClassifier([("lr",log_reg),("svc",svc),("knn",knn)])
pipe = Pipeline([("scale",StandardScaler()),("vc",vc)])
pipe.fit(X_train,y_train)
y_pred=pipe.predict(X_test)
accuracy_score(y_test,y_pred)

0.8022388059701493

# 6) StackingClassifier

In [20]:
st = StackingClassifier([("lr",log_reg),("svc",svc),("knn",knn)],final_estimator=LogisticRegression())
pipe = Pipeline([("scale",StandardScaler()),("vc",st)])
pipe.fit(X_train,y_train)
y_pred=pipe.predict(X_test)
accuracy_score(y_test,y_pred)

0.7947761194029851

In [21]:
with open("model.pkl","wb") as f:
    pkl.dump(vc,f)

In [32]:
with open("oneHotEncoder.pkl","wb") as f:
    pkl.dump(ohe,f)

In [30]:
X_train["Fare"].min()

0.0

In [23]:
df

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_Q,Embarked_S
0,0,3,1,22.000000,1,0,7.2500,0.0,1.0
1,1,1,0,38.000000,1,0,71.2833,0.0,0.0
2,1,3,0,26.000000,0,0,7.9250,0.0,1.0
3,1,1,0,35.000000,1,0,53.1000,0.0,1.0
4,0,3,1,35.000000,0,0,8.0500,0.0,1.0
...,...,...,...,...,...,...,...,...,...
886,0,2,1,27.000000,0,0,13.0000,0.0,1.0
887,1,1,0,19.000000,0,0,30.0000,0.0,1.0
888,0,3,0,29.699118,1,2,23.4500,0.0,1.0
889,1,1,1,26.000000,0,0,30.0000,0.0,0.0


In [25]:
X_train

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_Q,Embarked_S
445,1,1,4.000000,0,2,81.8583,0.0,1.0
650,3,1,29.699118,0,0,7.8958,0.0,1.0
172,3,0,1.000000,1,1,11.1333,0.0,1.0
450,2,1,36.000000,1,2,27.7500,0.0,1.0
314,2,1,43.000000,1,1,26.2500,0.0,1.0
...,...,...,...,...,...,...,...,...
106,3,0,21.000000,0,0,7.6500,0.0,1.0
270,1,1,29.699118,0,0,31.0000,0.0,1.0
860,3,1,41.000000,2,0,14.1083,0.0,1.0
435,1,0,14.000000,1,2,120.0000,0.0,1.0


In [36]:
ohe.transform([["Q"]])[0]

C:\Users\hp\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


array([1., 0.])